# Canada CPI — Data Exploration

This notebook demonstrates the data service layer using Canada-wide CPI data
from Statistics Canada (table 18-10-0004-13, 2002=100 baseline).

**Before running this notebook**, populate the local data cache:

```bash
uv run python scripts/fetch_cpi.py
```

After that, no network calls are made — the notebook reads entirely from the
local cache, which is what the `CutoffEnforcer` discipline requires.

In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from aieng.forecasting.data import DataService, SeriesMetadata
from aieng.forecasting.data.adapters import StatCanAdapter
from aieng.forecasting.evaluation import ForecastingTask

## 1. Build the DataService

Register Canada-wide CPI series. This reads from the local stats-can cache —
run `scripts/fetch_cpi.py` first if the cache is empty.

In [ ]:
CPI_TABLE_ID = "18-10-0004-13"
CACHE_DIR = Path("../../data/statcan")

CPI_SERIES = [
    ("cpi_all_items_canada",    "All-items",             "CPI All-items, Canada (2002=100)"),
    ("cpi_food_canada",         "Food",                  "CPI Food, Canada (2002=100)"),
    ("cpi_shelter_canada",      "Shelter",               "CPI Shelter, Canada (2002=100)"),
    ("cpi_transportation_canada", "Transportation",      "CPI Transportation, Canada (2002=100)"),
]

svc = DataService()

for series_id, product_group, description in CPI_SERIES:
    adapter = StatCanAdapter(
        table_id=CPI_TABLE_ID,
        member_filter={"GEO": "Canada", "Products and product groups": product_group},
        cache_dir=CACHE_DIR,
    )
    metadata = SeriesMetadata(
        series_id=series_id,
        description=description,
        source="StatCan",
        units="Index 2002=100",
        frequency="MS",
        table_id=CPI_TABLE_ID,
    )
    svc.register(series_id, adapter, metadata)
    print(f"  Registered: {series_id}")

print("\nDone.")

## 2. Inspect the registered series

In [ ]:
summary = svc.summary()
summary["start"] = summary["start"].dt.strftime("%Y-%m")
summary["end"]   = summary["end"].dt.strftime("%Y-%m")
summary

## 3. Cutoff filtering — the core discipline

Calling `get_series(series_id, as_of=...)` returns only observations that
were available on or before `as_of`. This is how backtests and live
forecasts share the same code path — the `as_of` date is the only
difference.

In [ ]:
# Retrieve All-items CPI as of 2023-01-01.
# No data from January 2023 onward will appear.
cutoff = datetime(2023, 1, 1)
df_cutoff = svc.get_series("cpi_all_items_canada", as_of=cutoff)

print(f"as_of: {cutoff.date()}")
print(f"Rows returned: {len(df_cutoff)}")
print(f"Latest observation: {df_cutoff['timestamp'].max().date()}")
df_cutoff.tail()

## 4. Plot: full CPI history by component

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

as_of_full = datetime(2026, 1, 1)  # include everything available

for series_id, _, description in CPI_SERIES:
    df = svc.get_series(series_id, as_of=as_of_full)
    ax.plot(df["timestamp"], df["value"], label=description, linewidth=1.5)

ax.set_title("Canada CPI by Component (2002=100)", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Index (2002=100)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Define a ForecastingTask

A `ForecastingTask` specifies the prediction problem without prescribing how
a predictor should solve it. Here we define the canonical 1-month-ahead
All-items CPI task.

In [ ]:
task = ForecastingTask(
    task_id="cpi_all_items_1m_ahead",
    target_series_id="cpi_all_items_canada",
    horizon=1,
    frequency="MS",
    description=(
        "Forecast Canada All-items CPI (2002=100) one month ahead. "
        "Resolution: observed CPI value at the target month-start timestamp."
    ),
)

print(task.model_dump_json(indent=2))

## 6. Year-over-year change

Derived from the index levels — illustrating that storing levels and
computing changes on-demand is the right approach.

In [ ]:
df_all = svc.get_series("cpi_all_items_canada", as_of=datetime(2026, 1, 1))
df_all = df_all.set_index("timestamp").sort_index()
df_all["yoy_pct"] = df_all["value"].pct_change(12) * 100

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_all.index, df_all["yoy_pct"], color="steelblue", linewidth=1.5)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Canada All-items CPI — Year-over-Year Change (%)", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("YoY change (%)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()